# 01 Data loading

Load the two training sets from TDC: `CYP3A4_Veith` for Model A (does a compound inhibit CYP3A4) and `hERG` for Model B (does a compound block the hERG channel). Check the size and the label balance of each, because the hERG set is small and uneven and that decides how we have to validate Model B.

Then convert both to Morgan fingerprints with `src/features.py` and save them, so the later notebooks do not have to featurise again. This notebook runs end to end.

In [1]:
import sys
import os

# Let the notebook import the helpers that live in src/.
sys.path.append(os.path.abspath(os.path.join("..", "src")))

import numpy as np
import pandas as pd
import features

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("fingerprint setting: radius", features.FP_RADIUS, "size", features.FP_SIZE)

fingerprint setting: radius 2 size 2048


## Get the data

TDC publishes its datasets as plain tab separated files on Harvard Dataverse. We download
them directly rather than using the `pytdc` package, which pins older numpy, pandas and
rdkit versions than the ones installed here, and which fails to build on Windows.

`data/` is gitignored, so this cell fetches the files the first time the notebook runs on
a new machine and skips them afterwards.

In [2]:
import urllib.request

os.makedirs(RAW_DIR, exist_ok=True)

TDC_FILES = {
    "cyp3a4_veith.tab": "https://dataverse.harvard.edu/api/access/datafile/4259582",
    "herg.tab": "https://dataverse.harvard.edu/api/access/datafile/4259588",
}


def download(url, destination):
    # Dataverse rejects the default urllib user agent with HTTP 403, so we send
    # an ordinary browser style one instead.
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        data = response.read()
    # Write only after the whole download succeeded, so an interrupted run never
    # leaves a half written file that looks valid next time.
    with open(destination, "wb") as handle:
        handle.write(data)
    return len(data)


for filename, url in TDC_FILES.items():
    destination = os.path.join(RAW_DIR, filename)
    if os.path.exists(destination):
        print("already have", filename)
        continue
    print("downloading", filename, "...")
    size = download(url, destination)
    print("   saved", size, "bytes to", destination)

downloading cyp3a4_veith.tab ...


   saved 746183 bytes to ..\data\raw\cyp3a4_veith.tab
already have herg.tab


## Model A data: CYP3A4 inhibition

In [3]:
# TDC publishes its datasets as plain tab separated files on Harvard Dataverse.
# We read them with pandas instead of the tdc package, because that package pins
# older numpy, pandas and rdkit versions than the ones installed here.
# Downloaded from: https://dataverse.harvard.edu/api/access/datafile/4259582
cyp_df = pd.read_csv(os.path.join(RAW_DIR, "cyp3a4_veith.tab"), sep="	")

print("CYP3A4_Veith shape:", cyp_df.shape)
print("columns:", list(cyp_df.columns))
cyp_df.head()

CYP3A4_Veith shape: (12328, 3)
columns: ['Drug_ID', 'Drug', 'Y']


,Drug_ID,Drug,Y
0,6602638.0,CCCC(=O)Nc1ccc(N2CCN(CC)CC2)c(Cl)c1.Cl,0
1,644510.0,O=c1[nH]c2cc3c(cc2cc1CN(CCCO)Cc1nnnn1Cc1ccc(F)...,1
2,644675.0,CC(=O)N(c1ccc2oc(=O)sc2c1)S(=O)(=O)c1cccs1,1
3,644890.0,COc1ccccc1C(c1nnnn1C(C)(C)C)N1CCN(Cc2ccncc2)CC1,1
4,645063.0,CC(=O)Nc1cccc(NC(=O)C2CCCN2C(=O)Nc2ccccc2C)c1,0


In [4]:
# Label balance decides whether plain accuracy would be misleading.
print("CYP3A4 label counts:")
print(cyp_df["Y"].value_counts())
print("positive fraction:", round(float(cyp_df["Y"].mean()), 3))

CYP3A4 label counts:
Y
0    7218
1    5110
Name: count, dtype: int64
positive fraction: 0.415


## Model B data: hERG block

In [5]:
# Downloaded from: https://dataverse.harvard.edu/api/access/datafile/4259588
# Only a few hundred compounds, which is why Model B needs a scaffold split
# or cross validation rather than one random holdout.
herg_df = pd.read_csv(os.path.join(RAW_DIR, "herg.tab"), sep="	")

print("hERG shape:", herg_df.shape)
print("columns:", list(herg_df.columns))
print("hERG label counts:")
print(herg_df["Y"].value_counts())
print("positive fraction:", round(float(herg_df["Y"].mean()), 3))

hERG shape: (655, 3)
columns: ['Drug_ID', 'Drug', 'Y']
hERG label counts:
Y
1.0    451
0.0    204
Name: count, dtype: int64
positive fraction: 0.689


## Features

`src/features.py` turns each molecule into 2058 numbers: 2048 Morgan fingerprint
**counts** (how many times each chemical fragment occurs, radius 2) followed by 10
whole-molecule descriptors such as molecular weight, logP and polar surface area.

Counts rather than plain yes/no bits, and the extra descriptors, were both chosen by
testing them on a validation split in notebook 02. They lifted Model A from 0.76 to 0.80
ROC-AUC on a scaffold split.

Some SMILES in public sets fail to parse, so `smiles_list_to_matrix` also returns the row
indices that survived. We use those indices to keep the labels lined up with the rows.

In [6]:
X_cyp, kept_cyp = features.smiles_list_to_matrix(cyp_df["Drug"].tolist())
y_cyp = cyp_df["Y"].values[kept_cyp]

print("CYP features:", X_cyp.shape, "=", features.FP_SIZE, "fragment counts +",
      features.N_DESCRIPTORS, "descriptors")
print("dropped as unparseable:", len(cyp_df) - len(kept_cyp))

# Distinct fragments per molecule, counting each one once. Looking at the raw
# feature sum would be meaningless here, because it would add molecular weight
# to a count of chemical fragments.
bits = features.bits_from_matrix(X_cyp)
print("mean distinct fragments per molecule:", round(float(bits.sum(axis=1).mean()), 1))

CYP features: (12328, 2058) = 2048 fragment counts + 10 descriptors
dropped as unparseable: 0
mean distinct fragments per molecule: 43.7


In [7]:
X_herg, kept_herg = features.smiles_list_to_matrix(herg_df["Drug"].tolist())
y_herg = herg_df["Y"].values[kept_herg]

print("hERG features:", X_herg.shape)
print("dropped as unparseable:", len(herg_df) - len(kept_herg))

bits_herg = features.bits_from_matrix(X_herg)
print("mean distinct fragments per molecule:", round(float(bits_herg.sum(axis=1).mean()), 1))

hERG features: (655, 2058)
dropped as unparseable: 0
mean distinct fragments per molecule: 46.0


In [8]:
# Save so notebooks 02 and 03 can start straight from the matrices.
np.save(os.path.join(PROCESSED_DIR, "X_cyp3a4.npy"), X_cyp)
np.save(os.path.join(PROCESSED_DIR, "y_cyp3a4.npy"), y_cyp)
np.save(os.path.join(PROCESSED_DIR, "X_herg.npy"), X_herg)
np.save(os.path.join(PROCESSED_DIR, "y_herg.npy"), y_herg)

# Keep the SMILES too. SHAP and the fragment mapping need the molecules back.
cyp_df.iloc[kept_cyp].to_csv(os.path.join(PROCESSED_DIR, "cyp3a4_clean.csv"), index=False)
herg_df.iloc[kept_herg].to_csv(os.path.join(PROCESSED_DIR, "herg_clean.csv"), index=False)
print("saved to", PROCESSED_DIR)

saved to ..\data\processed
